In [0]:
%python
# --- 01_BRONZE_KAFKA_STREAM.ipynb ---
from pyspark.sql.functions import from_json, col, current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType, TimestampType

# 1. KAFKA CONNECTION (Confluent Cloud)
kafka_key = "IKUYQNGVEGQP6IAV"
kafka_secret = "cfltIIIVZnAVEjNONmgAzD4PO44VFZ5JQTUQ3Q+qA+G/1kG7jy5HolGdNHsSx2uQ"
bootstrap_server = "pkc-9q8rv.ap-south-2.aws.confluent.cloud:9092"

kafka_options = {
    "kafka.bootstrap.servers": bootstrap_server,
    "SubscribePattern": "PRISM*",
    "startingOffsets": "earliest",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": f"kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username='{kafka_key}' password='{kafka_secret}';"
}

# 2. DEFINE JSON SCHEMA (matching the transaction structure)
json_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("amount", DecimalType(18, 2), True),
    StructField("country", StringType(), True),
    StructField("counterparty", StringType(), True),
    StructField("timestamp", TimestampType(), True)
])

# 3. READ STREAM AND PARSE JSON
df_stream = (spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
    .selectExpr("CAST(value AS STRING) as json_string")
    .select(from_json(col("json_string"), json_schema).alias("data"))
    .select(
        col("data.transaction_id"),
        col("data.user_id"),
        col("data.amount"),
        col("data.country"),
        col("data.counterparty"),
        col("data.timestamp"),
        lit("KAFKA_STREAM").alias("ingestion_source")
    ))

# 4. WRITE TO BRONZE TABLE
target_table = "`prism-sentinel-stream`.prism_bronze.transactions_raw"
checkpoint_path = "/Volumes/prism-sentinel-stream/prism_bronze/checkpoints/kafka_bronze"

print(f"🛰️ Sentinel listening for Kafka events on {bootstrap_server}...")

query = (df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(target_table))

In [0]:
%python
# --- 01_BRONZE_KAFKA_STREAM.ipynb ---
from pyspark.sql.functions import from_json, col, current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType, TimestampType

# 1. KAFKA CONNECTION (Confluent Cloud)
kafka_key = "IKUYQNGVEGQP6IAV"
kafka_secret = "cfltIIIVZnAVEjNONmgAzD4PO44VFZ5JQTUQ3Q+qA+G/1kG7jy5HolGdNHsSx2uQ"
bootstrap_server = "pkc-9q8rv.ap-south-2.aws.confluent.cloud:9092"

kafka_options = {
    "kafka.bootstrap.servers": bootstrap_server,
    "subscribePattern": "PRISM*",
    "startingOffsets": "earliest",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": f"kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username='{kafka_key}' password='{kafka_secret}';"
}

# 2. DEFINE JSON SCHEMA (matching the transaction structure)
json_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("amount", DecimalType(18, 2), True),
    StructField("country", StringType(), True),
    StructField("counterparty", StringType(), True),
    StructField("timestamp", TimestampType(), True)
])

# 3. READ STREAM AND PARSE JSON
df_stream = (spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
    .selectExpr("CAST(value AS STRING) as json_string")
    .select(from_json(col("json_string"), json_schema).alias("data"))
    .select(
        col("data.transaction_id"),
        col("data.user_id"),
        col("data.amount"),
        col("data.country"),
        col("data.counterparty"),
        col("data.timestamp"),
        lit("KAFKA_STREAM").alias("ingestion_source")
    ))

# 4. WRITE TO BRONZE TABLE
target_table = "`prism-sentinel-stream`.prism_bronze.transactions_raw"
checkpoint_path = "/Volumes/prism-sentinel-stream/prism_bronze/checkpoints/kafka_bronze"

print(f"🛰️ Sentinel listening for Kafka events on {bootstrap_server}...")

query = (df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(target_table))